In [1]:
import random

def generate_lighting_skus(num_ids=1143):
    # 1. Define the Vocabulary
    catalog_ids = ["S4PDMP", "T3PDMP", "S4TDMP", "Q4PDMP", "L2PDMP", "Z1PDMP", "S4QDP", "O3PDMP"]
    shapes = {
        "RPP": ["A", "B"], 
        "TPP": ["A", "B", "C"], 
        "TRI": ["A", "B", "C"], 
        "QDP": ["A", "B", "C", "D"], 
        "LPP": ["A", "B", "C"], 
        "ZPP": ["A", "B", "C", "D", "E"], 
        "OCT": ["A"]
    }
    angles = ["45C", "60C", "90C", "135C"]
    cri_list = [f"{i}CRI" for i in range(80, 101)]
    cct_list = ["27K", "30K", "35K", "40K", "50K"]
    lumen_list = ["400LMF", "600LMF", "800LMF", "900LMF", "1000LMF", "1200LMF"]
    voltage_list = ["MVOLT", "120V", "277V", "347V"]
    finishes = ["BLKT", "WHT"]
    sensors = ["SCT", "MIN1", "FLL", "PIRS", "PHLS", "THMS", "DTCH"]
    accessories = ["ZT", "RDCY", "APDT12", "SAPDT20", "F2/72A", "WHTCY", "WCR", "E10WLCP"]

    skus = set()

    while len(skus) < num_ids:
        # Pick Catalog ID and Shape
        cat_id = random.choice(catalog_ids)
        shape = random.choice(list(shapes.keys()))
        
        # Generate Dimensions with Inch/FT conversion
        dims = []
        for label in shapes[shape]:
            val_ft = random.randint(1, 12)
            if random.choice([True, False]):
                dims.append(f"{label}{val_ft}FT")
            else:
                dims.append(f"{label}{val_ft * 12}INCH")
        
        # Collect other tokens
        tokens = [
            random.choice(angles),
            random.choice(cri_list),
            random.choice(cct_list),
            random.choice(lumen_list),
            random.choice(voltage_list),
            random.choice(finishes)
        ]
        
        # Add a random number of sensors and accessories
        tokens.extend(random.sample(sensors, random.randint(1, 3)))
        tokens.extend(random.sample(accessories, random.randint(1, 4)))
        
        # Shuffle tokens (except first two) to mimic non-fixed positions
        random.shuffle(tokens)
        
        # Construct SKU
        sku_str = f"{cat_id} {shape} {' '.join(dims)} {' '.join(tokens)}"
        skus.add(sku_str)

    return list(skus)

# Execute and Save
generated_skus = generate_lighting_skus(1143)

# Save to a text file
with open("lighting_skus_dataset.txt", "w") as f:
    for i, sku in enumerate(generated_skus, 1):
        f.write(f"{sku}\n")

print(f"Successfully generated {len(generated_skus)} unique SKUs in 'lighting_skus_dataset.txt'")

Successfully generated 1143 unique SKUs in 'lighting_skus_dataset.txt'


In [2]:
import random
import math

def generate_lighting_skus(num_ids=1143):
    catalog_ids = ["S4PDMP", "T3PDMP", "S4TDMP", "Q4PDMP", "L2PDMP", "Z1PDMP", "S4QDP", "O3PDMP"]
    cri_list = [f"{i}CRI" for i in range(80, 101)]
    cct_list = ["27K", "30K", "35K", "40K", "50K"]
    lumen_list = ["400LMF", "600LMF", "800LMF", "900LMF", "1000LMF", "1200LMF"]
    voltage_list = ["MVOLT", "120V", "277V", "347V"]
    finishes = ["BLKT", "WHT"]
    sensors = ["SCT", "MIN1", "FLL", "PIRS", "PHLS", "THMS", "DTCH"]
    accessories = ["ZT", "RDCY", "APDT12", "SAPDT20", "F2/72A", "WHTCY", "WCR", "E10WLCP"]

    skus = set()

    while len(skus) < num_ids:
        cat_id = random.choice(catalog_ids)
        shape = random.choice(["RPP", "TPP", "TRI", "QDP", "LPP", "ZPP", "OCT"])
        
        dims = []
        angle_tokens = []

        # --- Geometric Logic Start ---
        
        if shape == "TRI":
            # Logic: If all sides are same, angle must be 60C 
            side_val = random.randint(2, 12)
            dims = [f"A{side_val}FT", f"B{side_val}FT", f"C{side_val}FT"]
            angle_tokens = ["60C"]

        elif shape in ["TPP", "LPP"]:
            # Logic: T and L shapes are perpendicular by definition 
            for label in (["A", "B", "C"] if shape == "TPP" else ["A", "B"]):
                dims.append(f"{label}{random.randint(2, 12)}FT")
            angle_tokens = ["90C"]

        elif shape == "RPP":
            # Logic: Rectangle (90C) or Parallelogram (angle1 + angle2 = 180)
            is_rectangle = random.choice([True, False])
            side_a = random.randint(2, 12)
            side_b = random.randint(2, 12)
            dims = [f"A{side_a}FT", f"B{side_b}FT"]
            if is_rectangle:
                angle_tokens = ["90C"]
            else:
                a1 = random.randint(30, 150)
                a2 = 180 - a1
                angle_tokens = [f"{a1}C", f"{a2}C"]

        elif shape == "QDP":
            # Logic: A=B=C=D (Square) -> 90C. If A=B, C=D -> 2 angles. Else 4 angles.
            side_a = random.randint(2, 12)
            side_b = random.randint(2, 12)
            if side_a == side_b: # Square Logic
                dims = [f"A{side_a}FT", f"B{side_a}FT"]
                angle_tokens = ["90C"]
            else: # Quadrilateral Logic
                dims = [f"A{side_a}FT", f"B{side_a}FT", f"C{side_b}FT", f"D{side_b}FT"]
                a1 = random.randint(45, 135)
                a2 = 180 - a1
                angle_tokens = [f"{a1}C", f"{a2}C"]

        elif shape == "ZPP":
            # Logic: 3 sides A,B,C and two angles between 5-130 
            dims = [f"A{random.randint(2, 10)}FT", f"B{random.randint(2, 10)}FT", f"C{random.randint(2, 10)}FT"]
            angle_tokens = [f"{random.randint(5, 130)}C", f"{random.randint(5, 130)}C"]

        elif shape == "OCT":
            # Logic: Octagon (sum 1080). Equal sides and 135C angles 
            dims = [f"A{random.randint(2, 8)}FT"]
            angle_tokens = ["135C"]

        # --- Geometric Logic End ---

        # Collect other tokens
        other_tokens = [
            random.choice(cri_list),
            random.choice(cct_list),
            random.choice(lumen_list),
            random.choice(voltage_list),
            random.choice(finishes)
        ]
        
        other_tokens.extend(random.sample(sensors, random.randint(1, 2)))
        other_tokens.extend(random.sample(accessories, random.randint(1, 2)))
        
        # Shuffle only non-primary tokens
        random.shuffle(other_tokens)
        
        # Construct SKU: ID + Shape + Dimensions + Angles + Others
        sku_str = f"{cat_id} {shape} {' '.join(dims)} {' '.join(angle_tokens)} {' '.join(other_tokens)}"
        skus.add(sku_str)

    return list(skus)

generated_skus = generate_lighting_skus(1143)

with open("lighting_skus_dataset.txt", "w") as f:
    for sku in generated_skus:
        f.write(f"{sku}\n")

print(f"Generated {len(generated_skus)} mathematically valid SKUs.")

Generated 1143 mathematically valid SKUs.
